# Step 8 (polish pass) — Combined Dashboard (tabs: Day/Night, Peak/Off-peak/Weekend, Daily Rhythm)

**This notebook produces the final interactive dashboard**, combining three topics into a single
file with tab navigation, and fixing UI issues found in earlier rounds.

**Input** (8 files):
- `output/monthly_day_night_summary.csv`
- `output/monthly_time_period_summary.csv`
- `output/monthly_temporal_ratios.csv`
- `output/station_month_day_night.csv`
- `output/station_month_time_period_summary.csv`
- `output/station_month_temporal_ratios.csv`
- `output/station_metadata.csv`
- `output/processed_hourly_bike_counts_time_categories.csv.gz` (used by the Daily Rhythm tab)

**Output**: `output/bike_dashboard.html` (a single self-contained file)

## What this version does

1. **Combined into one file with tab navigation**: three buttons at the top -- "Day / Night",
   "Peak / Off-peak / Weekend", "Daily Rhythm (24hr)". Clicking switches the visible content
   instead of scrolling through one long page. Implementation note: inactive tabs are hidden with
   CSS `display:none`; a Plotly chart rendered inside a hidden div has zero container width at
   first render, so switching to that tab also calls `Plotly.Plots.resize()` to force a recompute,
   avoiding squashed or blank charts.
2. **UI fixes**:
   - Panel titles moved out of the Plotly figure itself into an outer HTML `<h2>`, so they no
     longer overlap with the page buttons
   - Fixed a bug where the gradient legend was drawn twice -- now only one bar
   - All five seasonal-index maps (3 for day/night, 2 for peak/off-peak) uniformly set
     `showlegend=False`, so none of them can unexpectedly show a legend/numeric marker while the
     others don't
   - All explanatory text rewritten in plain language (no words like "straddle"/"skew"), using a
     consistent "How it's calculated" / "What it means" structure with short bullet-style
     sentences instead of long paragraphs
   - **Illustrative example numbers are now clearly separated from numbers computed from the
     data**: examples always start with "for example, a ratio of X would mean...", and
     data-derived summaries always state they are "the lowest to highest across the 14 months," so
     readers don't mistake either for a number they should be able to find directly on the chart
   - The linked map markers in Panel 2 / P2 were changed from red to green, so they don't get
     confused with the red/blue heatmap colorscale next to them
3. **Peak/Off-peak/Weekend tab**: structured like the Day/Night tab -- a network-wide monthly
   line chart, a station x month heatmap (left) + linked map (right), and two seasonal-index maps
   (`peak_offpeak_ratio`, `weekday_weekend_ratio`; volume maps were intentionally left out this
   round to avoid doing too much at once, can be added later if needed).
4. **Daily Rhythm (24hr) tab** (confirmed and finalized this round):
   - Panel R1: weekday vs weekend/holiday 24-hour profile (whole period, all stations combined) --
     answers "what does a typical day look like"
   - Panel R2: month x hour heatmap -- answers "does that shape change across the year"
   - This tab currently has **no station-level spatial dimension** (unlike the other two tabs,
     which each have a heatmap + map + seasonal-index trio) -- it's intentionally a lighter-weight
     version.

## Not done this round (decisions carried over from earlier)
- Bike lane network: on hold pending the team's discussion
- A filterable/sortable traffic-hotspot table: not included this round, a separate feature for later
- Mobile layout: only checked that common desktop widths (1280-1920px) don't break; no responsive
  layout work for phone screens


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.colors as pc
from plotly.subplots import make_subplots
from astral import LocationInfo
from astral.sun import sun

HAMBURG_LAT, HAMBURG_LON = 53.55, 10.00
N_GROUPS = 12
RELIABILITY_MIN_SAMPLE = 10
BASEMAP_STYLE = 'carto-positron'
OUTPUT_HTML = "output/bike_dashboard.html"

# --- added for Home + Summer vs Winter tabs ---
SEASONAL_STATION  = "output/station_seasonal_summer_winter.csv"
SEASONAL_PROFILE  = "output/seasonal_hourly_profile.csv"
NETWORK_SIMPLIFIED = "output/bike_network_simplified.geojson"

# Bike-network safety classes coloured by SAFETY rank (not by class number):
# Class I safest -> Class IV -> Class II -> Class III least safe.
SAFETY_CLASS_COLOR = {1: '#1a9850', 4: '#a6d96a', 2: '#f46d43', 3: '#a50026'}
SAFETY_CLASS_LABEL = {
    1: 'Class I \u00b7 Off-street path \u2014 safest',
    4: 'Class IV \u00b7 Structural cycle track \u2014 2nd',
    2: 'Class II \u00b7 Protected lane \u2014 3rd',
    3: 'Class III \u00b7 Mixed / bike lane \u2014 least safe',
}

NETWORK_CONTEXT = "output/bike_network_context.geojson"
BOUNDARY_SIMPLIFIED = "output/hamburg_boundary_simplified.geojson"


In [2]:
monthly_dn = pd.read_csv("output/monthly_day_night_summary.csv")
monthly_tp = pd.read_csv("output/monthly_time_period_summary.csv")
monthly_ratios = pd.read_csv("output/monthly_temporal_ratios.csv")
g_dn = pd.read_csv("output/station_month_day_night.csv")
g_tp = pd.read_csv("output/station_month_time_period_summary.csv")
g_tp_ratios = pd.read_csv("output/station_month_temporal_ratios.csv")
station_meta = pd.read_csv("output/station_metadata.csv")
# strip the legacy "(veraltet)" suffix present in every station name (display only; ids untouched)
def _clean_names(_df):
    if 'station_name' in _df.columns:
        _df['station_name'] = _df['station_name'].astype(str).str.replace(r'\s*\(veraltet\)\s*$', '', regex=True)
    return _df
for _df in (station_meta, g_dn, g_tp, g_tp_ratios):
    _clean_names(_df)
name_map = station_meta.set_index('station_id')['station_name'].to_dict()
coord_map = station_meta.set_index('station_id')[['longitude_wgs84', 'latitude_wgs84']].to_dict('index')
print("Data loaded")

Data loaded


In [3]:
def gradient_legend_html(colorscale_name, low_label, mid_label, high_label, low_emoji, mid_emoji, high_emoji):
    """A single gradient bar (fixes an earlier bug that drew it twice)"""
    stops = pc.get_colorscale(colorscale_name)
    css_stops = ', '.join(f"{c} {p*100:.0f}%" for p, c in stops)
    return f'''
    <div style="margin:8px 0;font-size:13px;">
      <div style="height:16px;border-radius:3px;background:linear-gradient(to right, {css_stops});"></div>
      <div style="display:flex;justify-content:space-between;margin-top:3px;">
        <span>{low_emoji} {low_label}</span>
        <span>{mid_emoji} {mid_label}</span>
        <span>{high_emoji} {high_label}</span>
      </div>
    </div>'''

def text_box(html_text):
    return f'<div class="explain">{html_text}</div>'

def panel_title(text):
    return f'<h2 class="panel-title">{text}</h2>'

## ===== TAB 1: Day / Night =====

In [4]:
# ---- D-Panel 1a: sunrise/sunset ----
loc = LocationInfo(name='Hamburg', region='Germany', timezone='Europe/Berlin',
                    latitude=HAMBURG_LAT, longitude=HAMBURG_LON)
dates = pd.date_range('2025-01-01', '2026-02-28', freq='D').date
sun_records = []
for d in dates:
    s = sun(loc.observer, date=d, tzinfo=loc.tzinfo)
    sr, ss = s['sunrise'], s['sunset']
    sun_records.append((d, sr.hour + sr.minute/60 + sr.second/3600, ss.hour + ss.minute/60 + ss.second/3600))
sun_df = pd.DataFrame(sun_records, columns=['date', 'sunrise_hour', 'sunset_hour'])

fig_d1a = go.Figure()
fig_d1a.add_trace(go.Scatter(x=sun_df['date'], y=sun_df['sunset_hour'], name='sunset',
                              line=dict(color='#8e44ad'), mode='lines'))
fig_d1a.add_trace(go.Scatter(x=sun_df['date'], y=sun_df['sunrise_hour'], name='sunrise',
                              line=dict(color='#f39c12'), mode='lines',
                              fill='tonexty', fillcolor='rgba(255,220,150,0.3)'))
fig_d1a.update_yaxes(title='Hour of day (0-24)', range=[0, 24])
fig_d1a.update_layout(height=420, autosize=True, hovermode='x unified', margin=dict(t=20))

TITLE_D1A = panel_title("Sunrise / Sunset time of day")
TEXT_D1A = text_box(
    "<b>How it's calculated:</b> actual sunrise/sunset time for Hamburg, computed day by day "
    "with the astral library.<br>"
    "<b>What it means:</b> the shaded band is daylight. This is the raw input used to split every "
    "hour into a 'day part' and a 'night part' everywhere else in this dashboard."
)

In [5]:
# ---- D-Panel 1b: daylight hours verification ----
months = monthly_dn['month'].tolist()
fig_d1b = make_subplots(specs=[[{"secondary_y": True}]])
fig_d1b.add_trace(go.Scatter(x=months, y=monthly_dn['day_hours_weighted'], name='day_hours_weighted',
                              line=dict(color='#f39c12', dash='dot'), mode='lines+markers'), secondary_y=False)
fig_d1b.add_trace(go.Scatter(x=months, y=monthly_dn['night_hours_weighted'], name='night_hours_weighted',
                              line=dict(color='#34495e', dash='dot'), mode='lines+markers'), secondary_y=False)
fig_d1b.add_trace(go.Scatter(x=months, y=monthly_dn['avg_day_count_per_hour'], name='avg_day_count_per_hour',
                              line=dict(color='#e67e22'), mode='lines+markers'), secondary_y=True)
fig_d1b.add_trace(go.Scatter(x=months, y=monthly_dn['avg_night_count_per_hour'], name='avg_night_count_per_hour',
                              line=dict(color='#2c3e50'), mode='lines+markers'), secondary_y=True)
fig_d1b.update_yaxes(title_text="Hours (denominator)", secondary_y=False)
fig_d1b.update_yaxes(title_text="Avg count per hour", secondary_y=True)
fig_d1b.update_layout(hovermode='x unified', height=420, autosize=True, margin=dict(t=20))

TITLE_D1B = panel_title("Daylight hours vs. average bike count")
TEXT_D1B = text_box(
    "<b>How it's calculated:</b> we don't count a whole hour as just 'day' or just 'night'. If "
    "sunrise happens partway through an hour, that hour is split -- part counted as day, part as "
    "night. day_hours_weighted adds up all the day-parts for the month. avg_day_count_per_hour = "
    "total daytime bike count &divide; day_hours_weighted, so a longer summer day does not "
    "automatically make this number bigger by itself.<br>"
    "<b>What it means:</b> looking across all 14 months, avg_day_count_per_hour ranges from about "
    "11 (its lowest month) to about 39 (its highest month) -- daytime traffic swings a lot with the "
    "seasons. avg_night_count_per_hour only ranges from about 4 to about 12 across the same 14 "
    "months -- nighttime traffic barely changes. Daytime biking looks weather/season-driven; "
    "nighttime biking -- likely commuters -- stays steady all year."
)

In [6]:
# ---- D-Panel 2: heatmap (left) + linked map (right), paginated ----
month_list = sorted(g_dn['month'].unique())
station_order_dn = g_dn.groupby('station_id')['avg_day_count_per_hour'].mean().sort_values(ascending=False).index.tolist()

pivot_ratio = g_dn.pivot(index='station_id', columns='month', values='day_night_ratio').reindex(station_order_dn)
pivot_day = g_dn.pivot(index='station_id', columns='month', values='avg_day_count_per_hour').reindex(station_order_dn)
pivot_night = g_dn.pivot(index='station_id', columns='month', values='avg_night_count_per_hour').reindex(station_order_dn)
pivot_reliable = g_dn.pivot(index='station_id', columns='month', values='day_night_ratio_reliable').reindex(station_order_dn)

log_all = np.log10(pivot_ratio.values.astype(float))
vmax_d2 = np.nanmax(np.abs(log_all[np.isfinite(log_all)]))

groups = np.array_split(station_order_dn, N_GROUPS)
fig_d2 = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
                        column_widths=[0.6, 0.4], horizontal_spacing=0.06)

TRACES_PER_GROUP = 2
buttons_d2 = []
for i, grp_stations in enumerate(groups):
    grp_stations = list(grp_stations)
    labels = [name_map.get(s, str(s)) for s in grp_stations]
    z = np.log10(pivot_ratio.reindex(grp_stations).values.astype(float))
    cd = np.dstack([
        np.tile(np.array(labels)[:, None], (1, len(month_list))),
        pivot_day.reindex(grp_stations).values,
        pivot_night.reindex(grp_stations).values,
        pivot_ratio.reindex(grp_stations).values,
        np.where(pivot_reliable.reindex(grp_stations).values == True, 'Yes',
                 np.where(pivot_reliable.reindex(grp_stations).values == False, 'No (too little data)', 'No data')),
    ])
    fig_d2.add_trace(go.Heatmap(
        z=z, x=month_list, y=labels, customdata=cd,
        colorscale='RdBu_r', zmid=0, zmin=-vmax_d2, zmax=vmax_d2, visible=(i == 0),
        hovertemplate=("Station: %{customdata[0]}<br>Month: %{x}<br>"
                        "avg_day: %{customdata[1]:.2f}<br>avg_night: %{customdata[2]:.2f}<br>"
                        "ratio: %{customdata[3]:.2f}<br>Enough data: %{customdata[4]}<extra></extra>"),
        colorbar=dict(title='ratio', x=0.56, tickmode='array', tickvals=[-vmax_d2, -vmax_d2/2, 0, vmax_d2/2, vmax_d2],
                      ticktext=[f'{10**t:.2f}' for t in [-vmax_d2, -vmax_d2/2, 0, vmax_d2/2, vmax_d2]]),
    ), row=1, col=1)

    lons = [coord_map[s]['longitude_wgs84'] for s in grp_stations]
    lats = [coord_map[s]['latitude_wgs84'] for s in grp_stations]
    fig_d2.add_trace(go.Scattermap(
        lon=lons, lat=lats, mode='markers', marker=dict(size=13, color='#2ecc71'),
        text=labels, hovertemplate="%{text}<extra></extra>", visible=(i == 0),
    ), row=1, col=2)

    vis = [False] * (N_GROUPS * TRACES_PER_GROUP)
    vis[i * TRACES_PER_GROUP] = True
    vis[i * TRACES_PER_GROUP + 1] = True
    r0 = sum(len(x) for x in groups[:i]) + 1
    r1 = r0 + len(grp_stations) - 1
    lbl = f'Rank {r0}-{r1}' + (' (busiest)' if i == 0 else ' (quietest)' if i == N_GROUPS - 1 else '')
    buttons_d2.append(dict(label=lbl, method='update', args=[{'visible': vis}]))

fig_d2.update_layout(
    updatemenus=[
        dict(type='buttons', direction='right', showactive=True, active=0, buttons=buttons_d2[:6],
             x=0.3, xanchor='center', y=1.12, yanchor='bottom', font=dict(size=9), pad=dict(l=2, r=2, t=2, b=2)),
        dict(type='buttons', direction='right', showactive=True, buttons=buttons_d2[6:],
             x=0.3, xanchor='center', y=1.05, yanchor='bottom', font=dict(size=9), pad=dict(l=2, r=2, t=2, b=2)),
    ],
    map=dict(style=BASEMAP_STYLE, zoom=9.8, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON)),
    height=780, autosize=True, xaxis_title='Month', yaxis=dict(tickfont=dict(size=10)),
    margin=dict(t=110),
)

TITLE_D2 = panel_title("Station map: Day/Night ratio by month")
TEXT_D2 = text_box(
    "<b>How it's calculated:</b> day_night_ratio = daytime average &divide; nighttime average, "
    "for each station each month. For example, a ratio of 3 would mean 3x more bikes were counted "
    "during the day than at night that month.<br>"
    "<b>Colors:</b> red = more day traffic, blue = more night traffic, white = about equal. "
    "Gray = not enough data that month (fewer than 10 total counts) to trust the number.<br>"
    "<b>What it means:</b> across the whole dataset, the middle station-month has a ratio of about "
    "3.3 -- most stations see roughly 3x more day traffic than night traffic. Use the buttons above "
    "to page through all 328 stations, ranked busiest to quietest. The map on the right shows where "
    "the stations on the current page are."
)

### D-Panel 3: seasonal index maps

In [7]:
g3_dn = g_dn.sort_values(['month', 'station_id']).copy()
g3_dn['total_traffic'] = g3_dn['total_day_count'] + g3_dn['total_night_count']

TITLE_D3_INTRO = panel_title("Seasonal maps: is each station busier or quieter than usual?")
TEXT_D3_INTRO = text_box(
    "<b>How it's calculated:</b> instead of raw traffic, these maps compare each station to "
    "ITSELF -- is this month busier or quieter than that station's own yearly average? 1x = normal "
    "for that station, 2x = twice as busy as usual, 0.5x = half as busy. Dot size = total traffic "
    "that month.<br>"
    "<b>What it means:</b> some stations are naturally 100x busier than others, so colouring by raw "
    "traffic would make quiet stations always look 'flat' even when their own season swings a lot. "
    "This way every station's seasonal pattern shows, however busy it normally is -- the trade-off "
    "is that colour no longer shows which stations are busiest overall (dot size still does).<br>"
    "<b>Note:</b> a few newly-installed stations had almost no traffic in their first months; those "
    "extreme values are colour-capped so they don't wash out the scale for everyone else."
)

In [8]:
# D-3a: ratio seasonal index
ratio_rel = g3_dn[g3_dn['day_night_ratio_reliable'] == True].copy()
ratio_baseline = ratio_rel.groupby('station_id')['day_night_ratio'].mean()
g3_dn['ratio_baseline'] = g3_dn['station_id'].map(ratio_baseline)
g3_dn['ratio_seasonal_index'] = g3_dn['day_night_ratio'] / g3_dn['ratio_baseline']
g3_dn['log_ratio_idx'] = np.log10(g3_dn['ratio_seasonal_index'])

map3a_df = g3_dn[g3_dn['day_night_ratio_reliable'] == True].copy()
vmax_r = map3a_df['log_ratio_idx'].abs().quantile(0.98)

fig_d3a = px.scatter_map(
    map3a_df, lat='latitude_wgs84', lon='longitude_wgs84',
    color='log_ratio_idx', size='total_traffic', hover_name='station_name',
    hover_data={'month': True, 'day_night_ratio': ':.2f', 'ratio_baseline': ':.2f',
                'ratio_seasonal_index': ':.2f',
                'latitude_wgs84': False, 'longitude_wgs84': False, 'log_ratio_idx': False, 'total_traffic': False},
    animation_frame='month', color_continuous_scale='RdBu_r', range_color=[-vmax_r, vmax_r],
    size_max=22, zoom=10.3, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON), map_style=BASEMAP_STYLE,
    height=600,
)
ticks_r = [-vmax_r, -vmax_r/2, 0, vmax_r/2, vmax_r]
fig_d3a.update_layout(coloraxis_colorbar=dict(title='x own average', tickmode='array',
                                               tickvals=ticks_r, ticktext=[f'{10**t:.2f}x' for t in ticks_r]),
                       autosize=True, margin=dict(t=20), showlegend=False)

TITLE_D3A = panel_title("3a. Day/Night ratio -- seasonal index")
LEGEND_D3A = gradient_legend_html('RdBu_r', 'more night-heavy than usual', 'typical', 'more day-heavy than usual',
                                   '🔵', '⚪', '🔴')
TEXT_D3A = text_box(
    "<b>How it's calculated:</b> each station-month's day/night ratio divided by that station's own "
    "average ratio, on a log scale centred at 1x.<br>"
    "<b>What it means:</b> colour shows whether a station's day/night balance was more day-heavy or "
    "more night-heavy than usual for that station -- not whether the station was busier overall."
)

In [9]:
# D-3b: day volume seasonal index (RdYlBu_r: red=above own avg, blue=below)
gg = g3_dn.copy()
gg['reliable'] = gg['total_day_count'] >= RELIABILITY_MIN_SAMPLE
baseline = gg[gg['reliable']].groupby('station_id')['avg_day_count_per_hour'].mean()
gg['baseline'] = gg['station_id'].map(baseline)
gg['seasonal_index'] = gg['avg_day_count_per_hour'] / gg['baseline']
gg['log_idx'] = np.log10(gg['seasonal_index'])
map_df = gg[gg['reliable']].copy()
vmax_v = map_df['log_idx'].abs().quantile(0.98)

fig_d3b = px.scatter_map(
    map_df, lat='latitude_wgs84', lon='longitude_wgs84',
    color='log_idx', size='total_traffic', hover_name='station_name',
    hover_data={'month': True, 'avg_day_count_per_hour': ':.2f', 'baseline': ':.2f', 'seasonal_index': ':.2f',
                'latitude_wgs84': False, 'longitude_wgs84': False, 'log_idx': False, 'total_traffic': False},
    animation_frame='month', color_continuous_scale='RdYlBu_r', range_color=[-vmax_v, vmax_v],
    size_max=22, zoom=10.3, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON), map_style=BASEMAP_STYLE,
    height=600,
)
ticks_v = [-vmax_v, -vmax_v/2, 0, vmax_v/2, vmax_v]
fig_d3b.update_layout(coloraxis_colorbar=dict(title='x own average', tickmode='array',
                                               tickvals=ticks_v, ticktext=[f'{10**t:.2f}x' for t in ticks_v]),
                       autosize=True, margin=dict(t=20), showlegend=False)

TITLE_D3B = panel_title("3b. Daytime traffic -- seasonal index")
LEGEND_D3B = gradient_legend_html('RdYlBu_r', 'quieter daytime than usual', 'typical', 'busier daytime than usual',
                                   '🔵', '🟡', '🔴')
TEXT_D3B = text_box(
    "<b>How it's calculated:</b> each station's daytime traffic for the month divided by that "
    "station's own usual daytime level, on a log scale centred at 1x.<br>"
    "<b>What it means:</b> red = busier daytime than usual for that station, blue = quieter than "
    "usual."
)

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [10]:
# D-3c: night volume seasonal index (PRGn_r: purple=above own avg, green=below)
gg = g3_dn.copy()
gg['reliable'] = gg['total_night_count'] >= RELIABILITY_MIN_SAMPLE
baseline = gg[gg['reliable']].groupby('station_id')['avg_night_count_per_hour'].mean()
gg['baseline'] = gg['station_id'].map(baseline)
gg['seasonal_index'] = gg['avg_night_count_per_hour'] / gg['baseline']
gg['log_idx'] = np.log10(gg['seasonal_index'])
map_df = gg[gg['reliable']].copy()
vmax_v = map_df['log_idx'].abs().quantile(0.98)

fig_d3c = px.scatter_map(
    map_df, lat='latitude_wgs84', lon='longitude_wgs84',
    color='log_idx', size='total_traffic', hover_name='station_name',
    hover_data={'month': True, 'avg_night_count_per_hour': ':.2f', 'baseline': ':.2f', 'seasonal_index': ':.2f',
                'latitude_wgs84': False, 'longitude_wgs84': False, 'log_idx': False, 'total_traffic': False},
    animation_frame='month', color_continuous_scale='PRGn_r', range_color=[-vmax_v, vmax_v],
    size_max=22, zoom=10.3, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON), map_style=BASEMAP_STYLE,
    height=600,
)
ticks_v = [-vmax_v, -vmax_v/2, 0, vmax_v/2, vmax_v]
fig_d3c.update_layout(coloraxis_colorbar=dict(title='x own average', tickmode='array',
                                               tickvals=ticks_v, ticktext=[f'{10**t:.2f}x' for t in ticks_v]),
                       autosize=True, margin=dict(t=20), showlegend=False)

TITLE_D3C = panel_title("3c. Nighttime traffic -- seasonal index")
LEGEND_D3C = gradient_legend_html('PRGn_r', 'quieter nighttime than usual', 'typical', 'busier nighttime than usual',
                                   '🟢', '⚪', '🟣')
TEXT_D3C = text_box(
    "<b>How it's calculated:</b> each station's nighttime traffic for the month divided by that "
    "station's own usual nighttime level, on a log scale centred at 1x.<br>"
    "<b>What it means:</b> purple = busier nighttime than usual for that station, green = quieter "
    "than usual."
)

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


## ===== TAB 2: Peak / Off-peak / Weekend =====

In [11]:
# ---- P-Panel 1: network-wide monthly lines (3 time periods) + ratio ----
pivot_tp = monthly_tp.pivot(index='month', columns='time_period', values='average_hourly_count').reindex(months)
colors_tp = {'weekday_peak': '#c0392b', 'weekday_offpeak': '#2980b9', 'weekend_public_holiday': '#27ae60'}
labels_tp = {'weekday_peak': 'weekday peak', 'weekday_offpeak': 'weekday off-peak', 'weekend_public_holiday': 'weekend / public holiday'}

fig_p1 = make_subplots(specs=[[{"secondary_y": True}]])
for col in ['weekday_peak', 'weekday_offpeak', 'weekend_public_holiday']:
    fig_p1.add_trace(go.Scatter(x=months, y=pivot_tp[col], name=labels_tp[col],
                                 line=dict(color=colors_tp[col]), mode='lines+markers'), secondary_y=False)
fig_p1.add_trace(go.Scatter(x=months, y=monthly_ratios['peak_offpeak_ratio'], name='peak/off-peak ratio',
                             line=dict(color='#8e44ad', dash='dot'), mode='lines+markers'), secondary_y=True)
fig_p1.update_yaxes(title_text="Avg count per hour", secondary_y=False)
fig_p1.update_yaxes(title_text="peak/off-peak ratio", secondary_y=True)
fig_p1.update_layout(hovermode='x unified', height=440, autosize=True, margin=dict(t=20))

TITLE_P1 = panel_title("Network-wide: weekday peak vs off-peak vs weekend/holiday")
TEXT_P1 = text_box(
    "<b>How it's calculated:</b> every hour is put into one of three groups -- weekday peak "
    "(06:00-09:00 and 16:00-19:00 commute hours), weekday off-peak (all other weekday hours), or "
    "weekend/public holiday (all day). We average bike counts within each group, per month, "
    "across all 328 stations.<br>"
    "<b>What it means:</b> weekday peak (red line) is always the busiest group, every month. Across "
    "the 14 months, off-peak and weekend traffic swing about 4x between their quietest and busiest "
    "month, but peak traffic only swings about 2x -- commuters keep a fairly steady rhythm while "
    "leisure riders come and go with the seasons. The dotted line (peak/off-peak ratio, right axis) "
    "is actually <i>higher in winter</i> (around 2.6-2.7) than summer (around 1.8) -- not because "
    "peak hours get busier in winter, but because off-peak/leisure riders mostly disappear then, "
    "making peak traffic look relatively bigger by comparison."
)

In [12]:
# ---- P-Panel 2: heatmap (left) + linked map (right), paginated ----
month_list_p = sorted(g_tp_ratios['month'].unique())
station_order_p = g_tp_ratios.groupby('station_id')['peak_share'].mean().sort_values(ascending=False).index.tolist()

pivot_p_ratio = g_tp_ratios.pivot(index='station_id', columns='month', values='peak_offpeak_ratio').reindex(station_order_p)
pivot_p_reliable = g_tp_ratios.pivot(index='station_id', columns='month', values='ratio_reliable').reindex(station_order_p)
pivot_p_wwratio = g_tp_ratios.pivot(index='station_id', columns='month', values='weekday_weekend_ratio').reindex(station_order_p)

log_all_p = np.log10(pivot_p_ratio.values.astype(float))
vmax_p2 = np.nanmax(np.abs(log_all_p[np.isfinite(log_all_p)]))

groups_p = np.array_split(station_order_p, N_GROUPS)
fig_p2 = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
                        column_widths=[0.6, 0.4], horizontal_spacing=0.06)

buttons_p2 = []
for i, grp_stations in enumerate(groups_p):
    grp_stations = list(grp_stations)
    labels = [name_map.get(s, str(s)) for s in grp_stations]
    z = np.log10(pivot_p_ratio.reindex(grp_stations).values.astype(float))
    cd = np.dstack([
        np.tile(np.array(labels)[:, None], (1, len(month_list_p))),
        pivot_p_ratio.reindex(grp_stations).values,
        pivot_p_wwratio.reindex(grp_stations).values,
        np.where(pivot_p_reliable.reindex(grp_stations).values == True, 'Yes',
                 np.where(pivot_p_reliable.reindex(grp_stations).values == False, 'No (too little data)', 'No data')),
    ])
    fig_p2.add_trace(go.Heatmap(
        z=z, x=month_list_p, y=labels, customdata=cd,
        colorscale='RdBu_r', zmid=0, zmin=-vmax_p2, zmax=vmax_p2, visible=(i == 0),
        hovertemplate=("Station: %{customdata[0]}<br>Month: %{x}<br>"
                        "peak/off-peak ratio: %{customdata[1]:.2f}<br>"
                        "weekday/weekend ratio: %{customdata[2]:.2f}<br>"
                        "Enough data: %{customdata[3]}<extra></extra>"),
        colorbar=dict(title='ratio', x=0.56, tickmode='array', tickvals=[-vmax_p2, -vmax_p2/2, 0, vmax_p2/2, vmax_p2],
                      ticktext=[f'{10**t:.2f}' for t in [-vmax_p2, -vmax_p2/2, 0, vmax_p2/2, vmax_p2]]),
    ), row=1, col=1)

    lons = [coord_map[s]['longitude_wgs84'] for s in grp_stations]
    lats = [coord_map[s]['latitude_wgs84'] for s in grp_stations]
    fig_p2.add_trace(go.Scattermap(
        lon=lons, lat=lats, mode='markers', marker=dict(size=13, color='#2ecc71'),
        text=labels, hovertemplate="%{text}<extra></extra>", visible=(i == 0),
    ), row=1, col=2)

    vis = [False] * (N_GROUPS * 2)
    vis[i * 2] = True
    vis[i * 2 + 1] = True
    r0 = sum(len(x) for x in groups_p[:i]) + 1
    r1 = r0 + len(grp_stations) - 1
    lbl = f'Rank {r0}-{r1}' + (' (most peak-heavy)' if i == 0 else ' (least peak-heavy)' if i == N_GROUPS - 1 else '')
    buttons_p2.append(dict(label=lbl, method='update', args=[{'visible': vis}]))

fig_p2.update_layout(
    updatemenus=[
        dict(type='buttons', direction='right', showactive=True, active=0, buttons=buttons_p2[:6],
             x=0.3, xanchor='center', y=1.12, yanchor='bottom', font=dict(size=9), pad=dict(l=2, r=2, t=2, b=2)),
        dict(type='buttons', direction='right', showactive=True, buttons=buttons_p2[6:],
             x=0.3, xanchor='center', y=1.05, yanchor='bottom', font=dict(size=9), pad=dict(l=2, r=2, t=2, b=2)),
    ],
    map=dict(style=BASEMAP_STYLE, zoom=9.8, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON)),
    height=780, autosize=True, xaxis_title='Month', yaxis=dict(tickfont=dict(size=10)),
    margin=dict(t=110),
)

TITLE_P2 = panel_title("Station map: peak/off-peak ratio by month")
TEXT_P2 = text_box(
    "<b>How it's calculated:</b> peak/off-peak ratio = weekday peak-hour average &divide; weekday "
    "off-peak average, for each station each month. For example, a ratio of 2 would mean peak-hour "
    "traffic was 2x off-peak traffic that month.<br>"
    "<b>Colors:</b> red = peak hours much busier than off-peak (strong commuter pattern), blue = "
    "peak and off-peak are similar or off-peak is busier (more leisure/all-day pattern). Gray = not "
    "enough data that month.<br>"
    "<b>What it means:</b> stations are ranked by how peak-heavy their traffic is on average. Use "
    "the buttons above to page through all 328 stations. The map on the right shows where the "
    "stations on the current page are."
)

### P-Panel 3: seasonal index maps

In [13]:
g3_p = g_tp_ratios.sort_values(['month', 'station_id']).copy()
# Use the sum of this station-month's 3 time_period totals as the dot size
# (same total_traffic concept used on the day/night side)
tp_total = g_tp.pivot(index=['station_id', 'month'], columns='time_period', values='total_count').reset_index()
tp_total['total_traffic'] = tp_total[['weekday_peak', 'weekday_offpeak', 'weekend_public_holiday']].sum(axis=1, skipna=True)
g3_p = g3_p.merge(tp_total[['station_id', 'month', 'total_traffic']], on=['station_id', 'month'], how='left')

TITLE_P3_INTRO = panel_title("Seasonal maps: peak/off-peak and weekday/weekend balance")
TEXT_P3_INTRO = text_box(
    "<b>How it's calculated:</b> same idea as the Day/Night tab -- each station is compared to "
    "ITSELF: is this month's commute pattern stronger or weaker than usual for that specific "
    "station? 1x = normal for that station. Dot size = total traffic that month.<br>"
    "<b>What it means:</b> this isolates each station's own seasonal shift in commuting, regardless "
    "of how busy the station is overall.<br>"
    "<b>Note:</b> a few newly-installed stations had almost no traffic in their first months; those "
    "extreme values are colour-capped so they don't wash out the scale for everyone else."
)

In [14]:
# P-3a: peak/off-peak ratio seasonal index
rel_p = g3_p[g3_p['ratio_reliable'] == True].copy()
baseline_p = rel_p.groupby('station_id')['peak_offpeak_ratio'].mean()
g3_p['po_baseline'] = g3_p['station_id'].map(baseline_p)
g3_p['po_seasonal_index'] = g3_p['peak_offpeak_ratio'] / g3_p['po_baseline']
g3_p['log_po_idx'] = np.log10(g3_p['po_seasonal_index'])

map_p3a_df = g3_p[g3_p['ratio_reliable'] == True].copy()
vmax_po = map_p3a_df['log_po_idx'].abs().quantile(0.98)

fig_p3a = px.scatter_map(
    map_p3a_df, lat='latitude_wgs84', lon='longitude_wgs84',
    color='log_po_idx', size='total_traffic', hover_name='station_name',
    hover_data={'month': True, 'peak_offpeak_ratio': ':.2f', 'po_baseline': ':.2f', 'po_seasonal_index': ':.2f',
                'latitude_wgs84': False, 'longitude_wgs84': False, 'log_po_idx': False, 'total_traffic': False},
    animation_frame='month', color_continuous_scale='RdBu_r', range_color=[-vmax_po, vmax_po],
    size_max=22, zoom=10.3, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON), map_style=BASEMAP_STYLE,
    height=600,
)
ticks_po = [-vmax_po, -vmax_po/2, 0, vmax_po/2, vmax_po]
fig_p3a.update_layout(coloraxis_colorbar=dict(title='x own average', tickmode='array',
                                               tickvals=ticks_po, ticktext=[f'{10**t:.2f}x' for t in ticks_po]),
                       autosize=True, margin=dict(t=20), showlegend=False)

TITLE_P3A = panel_title("3a. Peak/off-peak ratio -- seasonal index")
LEGEND_P3A = gradient_legend_html('RdBu_r', 'less peak-heavy than usual', 'typical', 'more peak-heavy than usual',
                                   '🔵', '⚪', '🔴')
TEXT_P3A = text_box(
    "<b>How it's calculated:</b> each station's peak/off-peak ratio for the month divided by that "
    "station's own average, on a log scale centred at 1x.<br>"
    "<b>What it means:</b> colour shows whether a station's commute pattern (peak vs off-peak) was "
    "stronger or weaker than usual for that station -- not whether the station was busier overall."
)

In [15]:
# P-3b: weekday/weekend ratio seasonal index
rel_w = g3_p[g3_p['ratio_reliable'] == True].copy()
baseline_w = rel_w.groupby('station_id')['weekday_weekend_ratio'].mean()
g3_p['ww_baseline'] = g3_p['station_id'].map(baseline_w)
g3_p['ww_seasonal_index'] = g3_p['weekday_weekend_ratio'] / g3_p['ww_baseline']
g3_p['log_ww_idx'] = np.log10(g3_p['ww_seasonal_index'])

map_p3b_df = g3_p[g3_p['ratio_reliable'] == True].copy()
vmax_ww = map_p3b_df['log_ww_idx'].abs().quantile(0.98)

fig_p3b = px.scatter_map(
    map_p3b_df, lat='latitude_wgs84', lon='longitude_wgs84',
    color='log_ww_idx', size='total_traffic', hover_name='station_name',
    hover_data={'month': True, 'weekday_weekend_ratio': ':.2f', 'ww_baseline': ':.2f', 'ww_seasonal_index': ':.2f',
                'latitude_wgs84': False, 'longitude_wgs84': False, 'log_ww_idx': False, 'total_traffic': False},
    animation_frame='month', color_continuous_scale='RdYlBu_r', range_color=[-vmax_ww, vmax_ww],
    size_max=22, zoom=10.3, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON), map_style=BASEMAP_STYLE,
    height=600,
)
ticks_ww = [-vmax_ww, -vmax_ww/2, 0, vmax_ww/2, vmax_ww]
fig_p3b.update_layout(coloraxis_colorbar=dict(title='x own average', tickmode='array',
                                               tickvals=ticks_ww, ticktext=[f'{10**t:.2f}x' for t in ticks_ww]),
                       autosize=True, margin=dict(t=20), showlegend=False)

TITLE_P3B = panel_title("3b. Weekday/weekend ratio -- seasonal index")
LEGEND_P3B = gradient_legend_html('RdYlBu_r', 'more weekend-heavy than usual', 'typical', 'more weekday-heavy than usual',
                                   '🔵', '🟡', '🔴')
TEXT_P3B = text_box(
    "<b>How it's calculated:</b> each station's weekday/weekend ratio for the month divided by that "
    "station's own average, on a log scale centred at 1x.<br>"
    "<b>What it means:</b> colour shows whether a station leaned more toward weekday traffic or more "
    "toward weekend/holiday traffic than usual for that station that month."
)

## ===== TAB 3: Daily Rhythm (24hr) =====

In [16]:
df_hourly = pd.read_csv(
    "output/processed_hourly_bike_counts_time_categories.csv.gz",
    dtype={'station_id': 'int64', 'month': 'str', 'time_period': 'str',
           'bike_count_hourly': 'float64'},
    usecols=['station_id', 'month', 'time_period', 'hour_24', 'bike_count_hourly'],
)
# hour_24 is stored without zero-padding in the source file (0,1,2...23, not 00,01,02...23);
# convert to int here and format for display separately, instead of relying on the raw string format
df_hourly['hour_24'] = df_hourly['hour_24'].astype(int)
print(f"Loaded: {len(df_hourly):,} rows")
print("Actual hour_24 value range:", sorted(df_hourly['hour_24'].unique()))

Loaded: 3,227,633 rows
Actual hour_24 value range: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]


In [17]:
# ---- R-Panel 1: weekday vs weekend/holiday 24hr profile ----
df_hourly['day_group'] = np.where(df_hourly['time_period'] == 'weekend_public_holiday',
                                   'weekend / public holiday', 'weekday')
r1 = df_hourly.groupby(['day_group', 'hour_24'])['bike_count_hourly'].mean().unstack(level=0)
r1 = r1.reindex(range(24))
hour_labels = [f'{h:02d}:00' for h in range(24)]

fig_r1 = go.Figure()
fig_r1.add_trace(go.Scatter(x=hour_labels, y=r1['weekday'], name='weekday', mode='lines+markers',
                             line=dict(color='#c0392b')))
fig_r1.add_trace(go.Scatter(x=hour_labels, y=r1['weekend / public holiday'], name='weekend / public holiday',
                             mode='lines+markers', line=dict(color='#27ae60')))
fig_r1.update_xaxes(title='Hour of day')
fig_r1.update_yaxes(title='Avg bike count')
fig_r1.update_layout(hovermode='x unified', height=440, autosize=True, margin=dict(t=20))

TITLE_R1 = panel_title("Weekday vs weekend/holiday -- 24-hour shape")
TEXT_R1 = text_box(
    "<b>How it's calculated:</b> average bike count for each hour of the day (0-23), split into "
    "weekday vs weekend/public holiday. This combines all 14 months and all 328 stations into one "
    "shape -- it answers 'what does a typical day look like', not 'does this change by month' (the "
    "heatmap below answers that).<br>"
    "<b>What it means:</b> weekdays show two clear humps -- a morning commute peak and a bigger "
    "afternoon peak. Weekends/holidays show one broad hump in the early afternoon, no sharp "
    "commute spikes. This is the daily 'shape' that the peak/off-peak hour boundaries "
    "(06:00-09:00, 16:00-19:00) were chosen to capture."
)

In [18]:
# ---- R-Panel 2: month x hour heatmap ----
r2 = df_hourly.groupby(['month', 'hour_24'])['bike_count_hourly'].mean().unstack(level=1)
month_order_r = sorted(r2.index)
r2 = r2.reindex(index=month_order_r, columns=range(24))

fig_r2 = go.Figure(data=go.Heatmap(
    z=r2.values, x=hour_labels, y=month_order_r, colorscale='YlOrRd',
    hovertemplate="Month: %{y}<br>Hour: %{x}<br>Avg count: %{z:.1f}<extra></extra>",
    colorbar=dict(title='avg count'),
))
fig_r2.update_xaxes(title='Hour of day')
fig_r2.update_yaxes(title='Month')
fig_r2.update_layout(height=460, autosize=True, margin=dict(t=20))

TITLE_R2 = panel_title("How the daily shape changes across the year")
TEXT_R2 = text_box(
    "<b>How it's calculated:</b> same as above, but broken out by month instead of collapsed into "
    "one line -- average bike count for each hour (0-23) of each month, network-wide.<br>"
    "<b>What it means:</b> the two-peak commute shape is visible in every month, but summer months "
    "(May-Sep) are hot across almost the whole day, not just the two peaks. Winter months (Nov-Feb) "
    "show a much sharper, narrower two-peak pattern with a clearly dark (quiet) stretch overnight "
    "and mid-morning. The commute pattern is stable year-round; the leisure/all-day traffic on top "
    "of it is what changes with the seasons."
)

## ===== TAB: Home (landing page) =====

In [19]:
# ---- Home tab: hero map (classified safety network + counting stations) ----
import json as _json
from collections import defaultdict as _dd

with open(NETWORK_SIMPLIFIED) as _f:
    _net = _json.load(_f)

# one line-string trace per safety class: concatenate segments, None-separated
_lines = _dd(lambda: {'lon': [], 'lat': []})
for _ft in _net['features']:
    _c = _ft['properties']['class_no']
    for _x, _y in _ft['geometry']['coordinates']:
        _lines[_c]['lon'].append(_x)
        _lines[_c]['lat'].append(_y)
    _lines[_c]['lon'].append(None)
    _lines[_c]['lat'].append(None)

# per-station total traffic (whole period) -> marker size
_st_traffic = (g_dn.assign(_tot=g_dn['total_day_count'] + g_dn['total_night_count'])
                   .groupby('station_id')['_tot'].sum())
_stations = station_meta[['station_id', 'station_name', 'longitude_wgs84', 'latitude_wgs84']].copy()
_stations['total_traffic'] = _stations['station_id'].map(_st_traffic).fillna(0)
_smax = _stations['total_traffic'].max()
_stations['msize'] = 5 + 16 * np.sqrt(_stations['total_traffic'] / _smax)   # sqrt so a few huge stations don't dominate

# intensity = period-wide average bikes per hour (fairer than cumulative total; used by the toggle)
_hh = g_dn.assign(_c=g_dn['total_day_count'] + g_dn['total_night_count'],
                  _h=g_dn['day_hours_weighted'] + g_dn['night_hours_weighted'])
_isum = _hh.groupby('station_id')[['_c', '_h']].sum()
_intensity = (_isum['_c'] / _isum['_h']).replace([np.inf, -np.inf], 0).fillna(0)
_stations['intensity'] = _stations['station_id'].map(_intensity).fillna(0)
_imax = _stations['intensity'].max()
_stations['msize_int'] = 5 + 16 * np.sqrt(_stations['intensity'] / _imax)

fig_home = go.Figure()

# draw least-safe first (bottom), safest last (top); legend reads safest -> least safe
_DRAW = [3, 2, 4, 1]
_RANK = {1: 1, 4: 2, 2: 3, 3: 4}
_WIDTH = {1: 2.0, 4: 2.0, 2: 2.0, 3: 1.0}
_OPACITY = {1: 0.95, 4: 0.95, 2: 0.95, 3: 0.5}
for _c in _DRAW:
    fig_home.add_trace(go.Scattermap(
        lon=_lines[_c]['lon'], lat=_lines[_c]['lat'], mode='lines',
        line=dict(width=_WIDTH[_c], color=SAFETY_CLASS_COLOR[_c]), opacity=_OPACITY[_c],
        name=SAFETY_CLASS_LABEL[_c], legendrank=_RANK[_c], hoverinfo='skip',
    ))

fig_home.add_trace(go.Scattermap(
    lon=_stations['longitude_wgs84'], lat=_stations['latitude_wgs84'], mode='markers',
    marker=dict(size=_stations['msize'].tolist(), color='#2c3e50'),
    name='Counting station', legendrank=5,
    text=_stations['station_name'],
    customdata=np.column_stack([_stations['total_traffic'], _stations['intensity']]),
    hovertemplate='<b>%{text}</b><br>total traffic: %{customdata[0]:,.0f}'
                  '<br>avg per hour: %{customdata[1]:.1f}<extra></extra>',
))

fig_home.update_layout(
    map=dict(style=BASEMAP_STYLE, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON), zoom=10.3),
    height=640, autosize=True, margin=dict(t=0, b=0, l=0, r=0), showlegend=True,
    legend=dict(bgcolor='rgba(255,255,255,0.85)', bordercolor='#ccc', borderwidth=1,
                x=0.01, y=0.99, xanchor='left', yanchor='top', font=dict(size=12)),
)

# toggle: size dots by total volume (default) or by average-per-hour intensity
fig_home.update_layout(updatemenus=[dict(
    type='buttons', direction='right', showactive=True, active=0,
    x=0.01, y=0.02, xanchor='left', yanchor='bottom',
    bgcolor='rgba(255,255,255,0.9)', bordercolor='#ccc', borderwidth=1, font=dict(size=11),
    buttons=[
        dict(label='Dot size: Total', method='restyle',
             args=[{'marker.size': [_stations['msize'].tolist()]}, [4]]),
        dict(label='Dot size: Avg/hr', method='restyle',
             args=[{'marker.size': [_stations['msize_int'].tolist()]}, [4]]),
    ],
)])

TITLE_HOME = panel_title("Hamburg cycling network \u2014 safety classes and counting stations")
TEXT_HOME = text_box(
    "<b>What this shows:</b> every bike-network segment coloured by its safety class (four levels, "
    "from the lab classification), with the 328 counting stations on top. Bigger dot = more total "
    "traffic over the whole period; use the buttons on the map to size dots by average bikes-per-hour instead.<br>"
    "<b>Safety classes (safest to least safe):</b> Class I off-street path, Class IV structural "
    "cycle track, Class II protected lane, Class III mixed traffic / bike lane. Click a legend "
    "entry to hide or isolate a class.<br>"
    "<b>Scope:</b> this dashboard currently covers the bike-traffic (N3) analysis only; the other "
    "Bike Safety Index indicators are not part of it yet."
)

NAV_CARDS = (
    '<div class="nav-grid">'
    '<div class="nav-card" onclick="showTab(\'tab-daynight\')">'
    '<div class="nav-card-title">Day / Night</div>'
    '<div class="nav-card-desc">How the day/night traffic balance shifts by station and month.</div></div>'
    '<div class="nav-card" onclick="showTab(\'tab-peak\')">'
    '<div class="nav-card-title">Peak / Off-peak / Weekend</div>'
    '<div class="nav-card-desc">Commuter peaks versus off-peak and weekend riding.</div></div>'
    '<div class="nav-card" onclick="showTab(\'tab-rhythm\')">'
    '<div class="nav-card-title">Daily Rhythm (24hr)</div>'
    '<div class="nav-card-desc">The 24-hour shape of a typical day, by month.</div></div>'
    '<div class="nav-card" onclick="showTab(\'tab-season\')">'
    '<div class="nav-card-title">Summer vs Winter</div>'
    '<div class="nav-card-desc">Which stations are seasonal, and how the day changes shape.</div></div>'
    '</div>'
)
print("Home tab built")


Home tab built


## ===== TAB: Summer vs Winter =====

In [20]:
# ---- Summer vs Winter: per-station ratio map ----
sw = pd.read_csv(SEASONAL_STATION)
sw['station_name'] = sw['station_name'].astype(str).str.replace(r'\s*\(veraltet\)\s*$', '', regex=True)
sw_rel = sw[sw['reliable'] == True].copy()
vmax_sw = float(np.nanquantile(sw_rel['log_swr'].abs(), 0.98))

fig_sw_map = px.scatter_map(
    sw_rel, lat='latitude_wgs84', lon='longitude_wgs84',
    color='log_swr', size='total_traffic', hover_name='station_name',
    hover_data={'summer_mean_hourly': ':.1f', 'winter_mean_hourly': ':.1f',
                'summer_winter_ratio': ':.2f',
                'latitude_wgs84': False, 'longitude_wgs84': False,
                'log_swr': False, 'total_traffic': False},
    color_continuous_scale='RdBu_r', range_color=[-vmax_sw, vmax_sw],
    size_max=22, zoom=10.3, center=dict(lat=HAMBURG_LAT, lon=HAMBURG_LON),
    map_style=BASEMAP_STYLE, height=600,
)
ticks_sw = [-vmax_sw, -vmax_sw / 2, 0, vmax_sw / 2, vmax_sw]
fig_sw_map.update_layout(
    coloraxis_colorbar=dict(title='summer / winter', tickmode='array',
                            tickvals=ticks_sw, ticktext=[f'{10**t:.2f}x' for t in ticks_sw]),
    autosize=True, margin=dict(t=20), showlegend=False,
)

TITLE_SW_MAP = panel_title("Which stations are the most seasonal?")
LEGEND_SW_MAP = gradient_legend_html('RdBu_r', 'more winter-heavy', 'similar both seasons',
                                     'more summer-heavy', '\U0001F535', '\u26AA', '\U0001F534')
TEXT_SW_MAP = text_box(
    "<b>How it's calculated:</b> for each station, mean hourly count in the summer window "
    "(2025-07-01 to 08-15) divided by mean hourly count in the winter window "
    "(2025-01-15 to 02-28). Shown on a log scale, centred on 1x (equal). Colour = the ratio; "
    "dot size = total traffic across both windows. For example, 2x means a station was twice as "
    "busy in summer as in winter.<br>"
    "<b>What it means:</b> almost every station is busier in summer than in winter (the median is "
    "about 1.7x), so the map is mostly red -- what matters is the degree. The deepest-red stations "
    "swing up the most in summer, behaving like leisure / recreational routes (waterfront, parks); "
    "paler stations stay closer to their winter level, more like year-round commuter routes; the "
    "rare blue stations are relatively winter-heavy. This relative view is the thing the "
    "month-by-month maps on the other tabs cannot show in a single glance.<br>"
    "<b>Note:</b> stations with fewer than 10 hourly records in either window, or missing from "
    "one window, are flagged and left off this map; a few extreme ratios are colour-capped so "
    "they do not wash out the scale (the true ratio still shows on hover)."
)
print("Summer vs Winter map built")


Summer vs Winter map built


In [21]:
# ---- Summer vs Winter: 24-hour profile ----
prof = pd.read_csv(SEASONAL_PROFILE)
hour_labels_sw = [f'{h:02d}:00' for h in range(24)]
_su = prof[prof['season'] == 'summer'].sort_values('hour')
_wi = prof[prof['season'] == 'winter'].sort_values('hour')

fig_sw_prof = go.Figure()
fig_sw_prof.add_trace(go.Scatter(x=hour_labels_sw, y=_su['mean_hourly'].values, name='summer',
                                 mode='lines+markers', line=dict(color='#e67e22')))
fig_sw_prof.add_trace(go.Scatter(x=hour_labels_sw, y=_wi['mean_hourly'].values, name='winter',
                                 mode='lines+markers', line=dict(color='#2980b9')))
fig_sw_prof.update_xaxes(title='Hour of day')
fig_sw_prof.update_yaxes(title='Avg bike count')
fig_sw_prof.update_layout(hovermode='x unified', height=440, autosize=True, margin=dict(t=20))

TITLE_SW_PROF = panel_title("Summer vs winter \u2014 24-hour shape")
TEXT_SW_PROF = text_box(
    "<b>How it's calculated:</b> average bike count for each hour of the day (0-23), across all "
    "stations, computed separately for the summer window and the winter window. Hours are read as "
    "local clock time (no timezone conversion).<br>"
    "<b>What it means:</b> the two seasons peak at different times of day. In this data the winter "
    "curve peaks in the morning commute hour, while the summer curve peaks in the mid-afternoon "
    "and stays high into the evening. The long daylight and leisure riding reshape the day, not "
    "just lift the totals. This is a different question from the weekday/weekend shape on the "
    "Daily Rhythm tab."
)
print("Summer vs Winter profile built")


Summer vs Winter profile built


## ===== Home add-on: City-wide hourly total (2025) =====

In [22]:
# ---- Home: city-wide hourly total over 2025 ----
_ct = pd.read_csv("output/processed_hourly_bike_counts_time_categories.csv.gz",
                  usecols=['hour_utc', 'year', 'bike_count_hourly'])
_ct = _ct[_ct['year'] == 2025].copy()
# hour_utc kept as local clock time (no tz_convert), consistent with the rest of the pipeline
_ct['ts'] = pd.to_datetime(_ct['hour_utc'], utc=True).dt.tz_localize(None)
_city = _ct.groupby('ts')['bike_count_hourly'].sum().sort_index().asfreq('h')
_roll_mean = _city.rolling(168, center=True, min_periods=24).mean()   # 7 days = 168 hours
_roll_med = _city.rolling(168, center=True, min_periods=24).median()

fig_city = go.Figure()
fig_city.add_trace(go.Scatter(x=_city.index, y=_city.values, name='Hourly total',
                              mode='lines', line=dict(color='#aac7e0', width=0.7)))
fig_city.add_trace(go.Scatter(x=_roll_med.index, y=_roll_med.values, name='7-day median',
                              mode='lines', line=dict(color='#7f8c8d', width=1.6)))
fig_city.add_trace(go.Scatter(x=_roll_mean.index, y=_roll_mean.values, name='7-day average',
                              mode='lines', line=dict(color='#c0392b', width=2.2)))
fig_city.update_xaxes(title='2025')
fig_city.update_yaxes(title='Bikes per hour (all stations)')
fig_city.update_layout(height=380, autosize=True, hovermode='x unified', margin=dict(t=30),
                       legend=dict(orientation='h', y=1.12, x=0))

TITLE_CITY = panel_title("City-wide bike traffic over 2025")
TEXT_CITY = text_box(
    "<b>How it's calculated:</b> every station's hourly count summed into one city-wide total per "
    "hour, across all of 2025. The two smooth lines are a 7-day rolling window (168 hours) over that "
    "hourly series -- the <b>average</b> and the <b>median</b> of the hours inside each window -- so "
    "day-to-day noise is smoothed into the seasonal trend. Hours are read as local clock time.<br>"
    "<b>What it means:</b> the thin line is the raw hourly pulse (the daily commute spikes); the "
    "rolling average follows the seasonal arc (higher in the warm months, lower in winter); the "
    "rolling median tracks the same arc but is less pulled by one-off extreme hours."
)
print("City Total (2025) built")


City Total (2025) built


## Assemble into a single HTML file (tab navigation + polished UI)

In [23]:
config = {'responsive': True}

# ---- shared JS consts for map underlays (grey context network + city boundary) ----
import json as _json2
def _flatten_multiline(_path):
    with open(_path) as _f:
        _fc = _json2.load(_f)
    _lo, _la = [], []
    for _ft in _fc['features']:
        _geom = _ft['geometry']
        _parts = _geom['coordinates'] if _geom['type'] == 'MultiLineString' else [_geom['coordinates']]
        for _ln in _parts:
            for _x, _y in _ln:
                _lo.append(_x); _la.append(_y)
            _lo.append(None); _la.append(None)
    return {'lon': _lo, 'lat': _la}
_CTX_JSON = _json2.dumps(_flatten_multiline(NETWORK_CONTEXT))
_BND_JSON = _json2.dumps(_flatten_multiline(BOUNDARY_SIMPLIFIED))
_UNDERLAY_JS = """
var __GREY_MAPS = ['pD3a','pD3b','pD3c','pP3a','pP3b','pSWmap'];
var __ALL_MAPS  = ['pHome','pD3a','pD3b','pD3c','pP3a','pP3b','pSWmap'];
function __ready(gd){ return gd && gd._fullLayout && gd.data; }
function __underlay(tries){
  tries = tries || 0;
  if(!window.Plotly){ if(tries<60){ setTimeout(function(){__underlay(tries+1);}, 200); } return; }
  var pending = false;
  __GREY_MAPS.forEach(function(id){
    var gd = document.getElementById(id);
    if(!__ready(gd)){ pending = true; return; }
    if(gd.__ctx) return;
    try{ Plotly.addTraces(gd, {type:'scattermap', subplot:'map', mode:'lines',
      lon:__CTX_NET.lon, lat:__CTX_NET.lat, line:{color:'#9aa0a6', width:0.6},
      opacity:0.4, hoverinfo:'skip', showlegend:false, name:'network (context)'});
      gd.__ctx = true; }catch(e){}
  });
  __ALL_MAPS.forEach(function(id){
    var gd = document.getElementById(id);
    if(!__ready(gd)){ pending = true; return; }
    if(gd.__bnd) return;
    try{ Plotly.addTraces(gd, {type:'scattermap', subplot:'map', mode:'lines',
      lon:__HH_BOUND.lon, lat:__HH_BOUND.lat, line:{color:'#5f6368', width:1.2},
      opacity:0.65, hoverinfo:'skip', showlegend:false, name:'city boundary'});
      gd.__bnd = true; }catch(e){}
  });
  if(pending && tries<60){ setTimeout(function(){__underlay(tries+1);}, 250); }
}
window.addEventListener('load', function(){ __underlay(0); });
"""

html_parts = []
html_parts.append('''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Hamburg Bike Count Dashboard</title>
<style>
html,body{margin:0;padding:0;width:100%;}
body{font-family:Arial,sans-serif;padding:20px 30px;box-sizing:border-box;}
h1{color:#2c3e50;margin-bottom:4px;}
.panel-title{color:#2c3e50;font-size:19px;font-weight:700;margin:10px 0 4px 0;}
.panel{margin-bottom:36px;border-bottom:1px solid #ddd;padding-bottom:20px;width:100%;}
.note{color:#666;font-size:14px;margin:10px 0;}
.explain{background:#f7f8fa;border-left:4px solid #7f8c8d;padding:12px 16px;
         margin:8px 0 18px 0;font-size:14px;line-height:1.6;color:#333;}
.explain b{color:#2c3e50;}
.js-plotly-plot{width:100% !important;}
.tab-bar{display:flex;gap:8px;margin:18px 0 22px 0;border-bottom:2px solid #ddd;flex-wrap:wrap;}
.tab-btn{padding:12px 24px;border:none;background:transparent;cursor:pointer;
         font-size:16px;font-weight:600;color:#888;border-bottom:3px solid transparent;
         transition:color 0.15s, border-color 0.15s;}
.tab-btn:hover{color:#2c3e50;}
.tab-btn.active{color:#2c3e50;border-bottom:3px solid #2c3e50;}
.tab-content{display:none;}
.tab-content.active{display:block;}
.nav-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:14px;margin:18px 0 8px 0;}
.nav-card{background:#f7f8fa;border:1px solid #e2e5e9;border-left:4px solid #2c3e50;border-radius:6px;
          padding:14px 16px;cursor:pointer;transition:box-shadow .15s, transform .05s;}
.nav-card:hover{box-shadow:0 3px 10px rgba(0,0,0,0.10);}
.nav-card:active{transform:translateY(1px);}
.nav-card-title{font-weight:700;color:#2c3e50;font-size:15px;margin-bottom:4px;}
.nav-card-desc{font-size:13px;color:#666;line-height:1.5;}
</style></head><body>
<h1>Hamburg Bike Count Dashboard</h1>
<p class="note">Data: 328 stations, 2025-01 to 2026-02. Use the tabs below to switch topic.</p>

<div class="tab-bar">
  <button id="btn-tab-home" class="tab-btn active" onclick="showTab('tab-home')">Home</button>
  <button id="btn-tab-daynight" class="tab-btn" onclick="showTab('tab-daynight')">Day / Night</button>
  <button id="btn-tab-peak" class="tab-btn" onclick="showTab('tab-peak')">Peak / Off-peak / Weekend</button>
  <button id="btn-tab-rhythm" class="tab-btn" onclick="showTab('tab-rhythm')">Daily Rhythm (24hr)</button>
  <button id="btn-tab-season" class="tab-btn" onclick="showTab('tab-season')">Summer vs Winter</button>
</div>
''')

# ---- Tab: Home (default active; carries the single Plotly.js CDN load) ----
html_parts.append('<div id="tab-home" class="tab-content active">')
html_parts.append('<div class="panel">')
html_parts.append(TITLE_HOME)
html_parts.append(fig_home.to_html(full_html=False, include_plotlyjs='cdn', div_id='pHome', config=config))
html_parts.append(TEXT_HOME)
html_parts.append('</div>')
html_parts.append('<div class="panel">')
html_parts.append(TITLE_CITY)
html_parts.append(fig_city.to_html(full_html=False, include_plotlyjs=False, div_id='pCity', config=config))
html_parts.append(TEXT_CITY)
html_parts.append('</div>')
html_parts.append(NAV_CARDS)
html_parts.append('</div>')  # end tab-home

# ---- Tab 1: Day / Night ----
html_parts.append('<div id="tab-daynight" class="tab-content">')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_D1A)
html_parts.append(fig_d1a.to_html(full_html=False, include_plotlyjs=False, div_id='pD1a', config=config))
html_parts.append(TEXT_D1A)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_D1B)
html_parts.append(fig_d1b.to_html(full_html=False, include_plotlyjs=False, div_id='pD1b', config=config))
html_parts.append(TEXT_D1B)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_D2)
html_parts.append(fig_d2.to_html(full_html=False, include_plotlyjs=False, div_id='pD2', config=config))
html_parts.append(TEXT_D2)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_D3_INTRO)
html_parts.append(TEXT_D3_INTRO)
html_parts.append(TITLE_D3A)
html_parts.append(fig_d3a.to_html(full_html=False, include_plotlyjs=False, div_id='pD3a', config=config))
html_parts.append(LEGEND_D3A)
html_parts.append(TEXT_D3A)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_D3B)
html_parts.append(fig_d3b.to_html(full_html=False, include_plotlyjs=False, div_id='pD3b', config=config))
html_parts.append(LEGEND_D3B)
html_parts.append(TEXT_D3B)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_D3C)
html_parts.append(fig_d3c.to_html(full_html=False, include_plotlyjs=False, div_id='pD3c', config=config))
html_parts.append(LEGEND_D3C)
html_parts.append(TEXT_D3C)
html_parts.append('</div>')

html_parts.append('</div>')  # end tab-daynight

# ---- Tab 2: Peak / Off-peak / Weekend ----
html_parts.append('<div id="tab-peak" class="tab-content">')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_P1)
html_parts.append(fig_p1.to_html(full_html=False, include_plotlyjs=False, div_id='pP1', config=config))
html_parts.append(TEXT_P1)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_P2)
html_parts.append(fig_p2.to_html(full_html=False, include_plotlyjs=False, div_id='pP2', config=config))
html_parts.append(TEXT_P2)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_P3_INTRO)
html_parts.append(TEXT_P3_INTRO)
html_parts.append(TITLE_P3A)
html_parts.append(fig_p3a.to_html(full_html=False, include_plotlyjs=False, div_id='pP3a', config=config))
html_parts.append(LEGEND_P3A)
html_parts.append(TEXT_P3A)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_P3B)
html_parts.append(fig_p3b.to_html(full_html=False, include_plotlyjs=False, div_id='pP3b', config=config))
html_parts.append(LEGEND_P3B)
html_parts.append(TEXT_P3B)
html_parts.append('</div>')

html_parts.append('</div>')  # end tab-peak

# ---- Tab 3: Daily Rhythm ----
html_parts.append('<div id="tab-rhythm" class="tab-content">')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_R1)
html_parts.append(fig_r1.to_html(full_html=False, include_plotlyjs=False, div_id='pR1', config=config))
html_parts.append(TEXT_R1)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_R2)
html_parts.append(fig_r2.to_html(full_html=False, include_plotlyjs=False, div_id='pR2', config=config))
html_parts.append(TEXT_R2)
html_parts.append('</div>')

html_parts.append('</div>')  # end tab-rhythm

# ---- Tab: Summer vs Winter ----
html_parts.append('<div id="tab-season" class="tab-content">')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_SW_MAP)
html_parts.append(fig_sw_map.to_html(full_html=False, include_plotlyjs=False, div_id='pSWmap', config=config))
html_parts.append(LEGEND_SW_MAP)
html_parts.append(TEXT_SW_MAP)
html_parts.append('</div>')

html_parts.append('<div class="panel">')
html_parts.append(TITLE_SW_PROF)
html_parts.append(fig_sw_prof.to_html(full_html=False, include_plotlyjs=False, div_id='pSWprof', config=config))
html_parts.append(TEXT_SW_PROF)
html_parts.append('</div>')

html_parts.append('</div>')  # end tab-season

html_parts.append('''
<script>
function showTab(id, btn) {
  document.querySelectorAll('.tab-content').forEach(el => el.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(el => el.classList.remove('active'));
  document.getElementById(id).classList.add('active');
  var tabBtn = document.getElementById('btn-' + id);
  if (tabBtn) tabBtn.classList.add('active');
  document.querySelectorAll('#' + id + ' .js-plotly-plot').forEach(el => {
    Plotly.Plots.resize(el);
  });
  window.scrollTo(0, 0);
}
</script>''')

html_parts.append('<script>\nvar __CTX_NET=' + _CTX_JSON + ';\nvar __HH_BOUND=' + _BND_JSON + ';\n' + _UNDERLAY_JS + '</script>')

html_parts.append('</body></html>')

with open(OUTPUT_HTML, 'w') as f:
    f.write(''.join(html_parts))

import os
print(f"Done: {OUTPUT_HTML} ({os.path.getsize(OUTPUT_HTML)/1e6:.2f} MB)")


Done: output/bike_dashboard.html (14.12 MB)
